# 03 - BloodMNIST Tuning

Notebook này chỉ load feature `.npy` của BloodMNIST và chạy tuning cho warm-up và lambda sweep.


In [1]:
import sys
import pandas as pd
import torch

sys.path.append('..')
from src.utils import get_device, load_feature_file, set_seed, train_clustering

set_seed(42)
device = get_device()
X, y = load_feature_file('../data/bloodmnist_resnet_features.npy', '../data/bloodmnist_labels.npy', device=device)
true_k = int(torch.unique(y).numel())
print('[INFO] BloodMNIST features:', tuple(X.shape), '| labels:', tuple(y.shape), '| K_true =', true_k)


[INFO] BloodMNIST features: (3421, 512) | labels: (3421,) | K_true = 8


In [2]:
def run_cfg(name, k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True, epochs=50):
    k_star, acc, nmi, ari = train_clustering(
        features=X,
        labels=y,
        k_max=k0,
        device=device,
        epochs=epochs,
        lr=1e-3,
        lambda_param=lambda_param,
        gamma=0.1,
        warmup_epochs=warmup_epochs,
        enable_split=enable_split,
        enable_merge=enable_merge,
    )
    return {
        'Setting': name,
        'K0': k0,
        'lambda': lambda_param,
        'warmup_epochs': warmup_epochs,
        'K*': int(k_star),
        '|K*-K_true|': abs(int(k_star) - true_k),
        'ACC(%)': acc * 100,
        'NMI(%)': nmi * 100,
        'ARI(%)': ari * 100,
    }


In [3]:
baseline_rows = [
    run_cfg('PnP dynamic', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True),
    run_cfg('Fixed-K baseline', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=False, enable_merge=False),
]
pd.DataFrame(baseline_rows)


,Setting,K0,lambda,warmup_epochs,K*,|K*-K_true|,ACC(%),NMI(%),ARI(%)
0,PnP dynamic,12,2.0,20,12,4,37.065186,38.135968,23.655364
1,Fixed-K baseline,12,2.0,20,12,4,38.263666,36.224433,22.519206


In [4]:
lambda_rows = []
for lam in [1.5, 2.0, 2.5, 3.0]:
    lambda_rows.append(run_cfg(f'lambda={lam}', k0=12, lambda_param=lam, warmup_epochs=20, enable_split=True, enable_merge=True))

pd.DataFrame(lambda_rows)


,Setting,K0,lambda,warmup_epochs,K*,|K*-K_true|,ACC(%),NMI(%),ARI(%)
0,lambda=1.5,12,1.5,20,12,4,35.632856,36.320273,21.794766
1,lambda=2.0,12,2.0,20,12,4,36.655949,36.409657,23.330981
2,lambda=2.5,12,2.5,20,5,3,41.391406,30.713625,22.483860
3,lambda=3.0,12,3.0,20,5,3,44.606840,32.502076,23.882156


In [5]:
warmup_rows = []
for warmup in [0, 10, 20, 30]:
    warmup_rows.append(run_cfg(f'warmup={warmup}', k0=12, lambda_param=2.0, warmup_epochs=warmup, enable_split=True, enable_merge=True))

pd.DataFrame(warmup_rows)


,Setting,K0,lambda,warmup_epochs,K*,|K*-K_true|,ACC(%),NMI(%),ARI(%)
0,warmup=0,12,2.0,0,7,1,38.877521,31.187455,19.463012
1,warmup=10,12,2.0,10,9,1,41.683718,36.805624,23.470148
2,warmup=20,12,2.0,20,10,2,39.666764,32.861908,21.870628
3,warmup=30,12,2.0,30,11,3,37.942122,35.554548,22.183309
